In [1]:
import json, math
import pandas as pd
import numpy as np

OFF = "outputs/taskc/gen_eval_official.jsonl"       # 你的 official_150w
EVI = "outputs/taskc/gen_eval_evidence.jsonl"       # 你的 evidence_prompt

def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line: 
                continue
            rows.append(json.loads(line))
    return rows

def first(x):
    return x[0] if isinstance(x,list) and len(x) else x

def find_rb_llm_key(metrics: dict):
    # 自动找 metrics 里包含 rb + llm 的 key
    for k in metrics.keys():
        lk=k.lower()
        if "rb" in lk and "llm" in lk:
            return k
    return None

def add_scores(rows):
    out=[]
    for r in rows:
        m = r.get("metrics") or {}
        rl_f   = first(m.get("RL_F"))
        rb_alg = first(m.get("RB_agg"))  # 你这里 RB_agg 基本等价 RB_alg
        rbk    = find_rb_llm_key(m)
        rb_llm = first(m.get(rbk)) if rbk else None

        # harmonic mean
        final=None
        if rl_f and rb_alg and rb_llm and min(rl_f, rb_alg, rb_llm) > 0:
            final = 3.0/(1.0/rl_f + 1.0/rb_llm + 1.0/rb_alg)

        out.append({
            "task_id": r.get("task_id"),
            "Collection": r.get("Collection"),
            "domain": r.get("domain"),
            "question_source": r.get("question_source"),
            "pred": (r.get("predictions") or [{}])[0].get("text"),
            "RL_F": rl_f,
            "RB_alg": rb_alg,
            "RB_llm": rb_llm,
            "final_hmean": final,
            # 尽量提取问题文本：有的文件是 input 字段，有的是 prompt_strict/prompt_official
            "input": r.get("input"),
        })
    return pd.DataFrame(out)

df_off = add_scores(load_jsonl(OFF)).set_index("task_id")
df_evi = add_scores(load_jsonl(EVI)).set_index("task_id")

df = df_off.add_prefix("off_").join(df_evi.add_prefix("evi_"), how="inner")
df.shape


(100, 18)

In [2]:
df["delta_final"] = df["evi_final_hmean"] - df["off_final_hmean"]
df["delta_RL_F"]  = df["evi_RL_F"] - df["off_RL_F"]
df["delta_RB_llm"]= df["evi_RB_llm"] - df["off_RB_llm"]
df["delta_RB_alg"]= df["evi_RB_alg"] - df["off_RB_alg"]

# 掉分最多 Top 30
worst = df.sort_values("delta_final").head(30)
worst[[
    "off_domain","off_Collection",
    "off_final_hmean","evi_final_hmean","delta_final",
    "off_RL_F","evi_RL_F","delta_RL_F",
    "off_RB_llm","evi_RB_llm","delta_RB_llm",
    "off_RB_alg","evi_RB_alg","delta_RB_alg",
]]


,off_domain,off_Collection,off_final_hmean,evi_final_hmean,delta_final,off_RL_F,evi_RL_F,delta_RL_F,off_RB_llm,evi_RB_llm,delta_RB_llm,off_RB_alg,evi_RB_alg,delta_RB_alg
task_id,,,,,,,,,,,,,,
4a8369b7403e54df94d7fa495a4eff27<::>4,None,mt-rag-fiqa-beir-elser-512-100-20240501,0.496431,0.290779,-0.205652,1.000000,0.500000,-0.500000,0.6,0.2,-0.4,0.296167,0.301467,0.005299
79f0d0539d9ec0acbf90cb3388b30c17<::>6,None,mt-rag-clapnq-elser-512-100-20240503,0.727377,0.521966,-0.205412,1.000000,0.500000,-0.500000,0.8,0.7,-0.1,0.533502,0.431233,-0.102269
0208bf26ec357a803445290fa88a2e9e<::>6,None,mt-rag-clapnq-elser-512-100-20240503,0.849127,0.653937,-0.195190,1.000000,0.875000,-0.125000,0.9,0.9,0.0,0.703270,0.428517,-0.274753
b2805ee4c478194c234e2384ffb0a6bf<::>4,None,mt-rag-clapnq-elser-512-100-20240503,0.798735,0.609147,-0.189588,1.000000,0.500000,-0.500000,1.0,1.0,0.0,0.569496,0.519503,-0.049993
c6c3b02ca32795af64c903dd76700517<::>2,None,mt-rag-ibmcloud-elser-512-100-20240502,0.736792,0.559436,-0.177356,0.875000,0.500000,-0.375000,1.0,1.0,0.0,0.518444,0.423273,-0.095172
d2432696b32af73cf3dabd7090997afb<::>5,None,mt-rag-govt-elser-512-100-20240611,0.780800,0.626177,-0.154623,1.000000,0.500000,-0.500000,0.9,1.0,0.1,0.577667,0.558355,-0.019312
5815bb4a99e2a0d8a986348da4c49083<::>8,None,mt-rag-ibmcloud-elser-512-100-20240502,0.706982,0.552724,-0.154258,0.875000,0.500000,-0.375000,0.9,0.9,0.0,0.502659,0.431677,-0.070982
941445ba11ba7ba2c92c5184c9d798d6<::>1,None,mt-rag-govt-elser-512-100-20240611,0.687203,0.533759,-0.153445,1.000000,0.500000,-0.500000,0.7,0.7,0.0,0.516276,0.456216,-0.060061
dd6b6ffd177f2b311abe676261279d2f<::>6,None,mt-rag-clapnq-elser-512-100-20240503,0.797028,0.643831,-0.153196,1.000000,1.000000,0.000000,1.0,1.0,0.0,0.566898,0.375996,-0.190903


In [3]:
def show_case(tid):
    row = df.loc[tid]
    print("task_id:", tid)
    print("domain:", row.get("off_domain"), "Collection:", row.get("off_Collection"))
    print("\nSCORES:")
    print("  official final:", row["off_final_hmean"], "  evidence final:", row["evi_final_hmean"], "  delta:", row["delta_final"])
    print("  RL_F   off/evi:", row["off_RL_F"], row["evi_RL_F"], "delta:", row["delta_RL_F"])
    print("  RB_llm off/evi:", row["off_RB_llm"], row["evi_RB_llm"], "delta:", row["delta_RB_llm"])
    print("  RB_alg off/evi:", row["off_RB_alg"], row["evi_RB_alg"], "delta:", row["delta_RB_alg"])

    inp = row.get("off_input")
    if isinstance(inp, list) and inp:
        # 很多时候 input 是 list[turns]；这里简单打印最后一段
        print("\nINPUT (truncated):")
        s = str(inp)[:2000]
        print(s)
    elif isinstance(inp, str):
        print("\nINPUT:")
        print(inp[:2000])

    print("\nOFFICIAL ANSWER:\n", (row.get("off_pred") or "")[:2000])
    print("\nEVIDENCE ANSWER:\n", (row.get("evi_pred") or "")[:2000])

# 看掉分最狠的前 5 个
for tid in worst.index[:5]:
    show_case(tid)
    print("\n" + "="*100 + "\n")


task_id: 4a8369b7403e54df94d7fa495a4eff27<::>4
domain: None Collection: mt-rag-fiqa-beir-elser-512-100-20240501

SCORES:
  official final: 0.49643104010795214   evidence final: 0.2907789312078199   delta: -0.20565210890013225
  RL_F   off/evi: 1.0 0.5 delta: -0.5
  RB_llm off/evi: 0.6 0.2 delta: -0.39999999999999997
  RB_alg off/evi: 0.2961674087 0.3014666844 delta: 0.005299275699999995

INPUT (truncated):
[{'speaker': 'user', 'text': 'how does pre-market trading work?', 'metadata': {'author_type': 'human', 'author_id': 'f064a81c-6653-44b4-8053-fb4da0e0250a', 'created_at': 1726254963}}, {'speaker': 'agent', 'text': 'Pre-market trading takes place before the regular trading hours (RTH) of the stock market. For example, pre-market hours are from 4:00 a.m. to 9:30 a.m. Eastern Time (ET) on the NASDAQ stock market. During pre-market hours, you can place limit orders, but you cannot place market orders due to lower liquidity. This is because the volume and liquidity are generally much lower

In [5]:
import json
from pathlib import Path

OFF = "outputs/taskc/gen_eval_official.jsonl"
EVI = "outputs/taskc/gen_eval_evidence.jsonl"

def load_claim_map(path):
    mp={}
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if not line.strip(): 
                continue
            r=json.loads(line)
            tid=r.get("task_id")
            m=r.get("metrics") or {}
            mp[tid]={
                "RL_F": (m.get("RL_F") or [None])[0] if isinstance(m.get("RL_F"), list) else m.get("RL_F"),
                "RL_F_claims": m.get("RL_F_claims"),
            }
    return mp

off_claims = load_claim_map(OFF)
evi_claims = load_claim_map(EVI)


In [6]:
import numpy as np

df2 = df.copy()

# 让 NaN 的 delta_final 用一个很差的值替代，避免排不出来
# 如果某条 final 缺失，通常是缺 RL_F/RB_llm/RB_alg，实际也是“坏样本”
df2["delta_final_rank"] = df2["delta_final"].fillna(-999)

worst3 = df2.sort_values("delta_final_rank").head(3)
worst3.index.tolist(), worst3[["delta_final","delta_RL_F","delta_RB_llm","delta_RB_alg"]]


(['be461bfeda2d4826cdb663dcaa7d1ced<::>4',
  '13a398b30067d58d179fa8cbc2592449<::>9',
  '4a8369b7403e54df94d7fa495a4eff27<::>4'],
                                        delta_final  delta_RL_F  delta_RB_llm  \
 task_id                                                                        
 be461bfeda2d4826cdb663dcaa7d1ced<::>4          NaN         0.0           0.0   
 13a398b30067d58d179fa8cbc2592449<::>9          NaN        -0.5           0.0   
 4a8369b7403e54df94d7fa495a4eff27<::>4    -0.205652        -0.5          -0.4   
 
                                        delta_RB_alg  
 task_id                                              
 be461bfeda2d4826cdb663dcaa7d1ced<::>4      0.042416  
 13a398b30067d58d179fa8cbc2592449<::>9      0.237790  
 4a8369b7403e54df94d7fa495a4eff27<::>4      0.005299  )

In [7]:
from collections import Counter

def flatten_claims(claims):
    flat=[]
    def walk(x):
        if isinstance(x, list):
            for y in x: walk(y)
        elif isinstance(x, dict) and "claim" in x and "label" in x:
            flat.append(x)
    walk(claims)
    return flat

def summarize_labels(flat):
    c=Counter([x.get("label") for x in flat])
    return dict(c)

def print_case(tid, max_claims=15, ans_chars=1500):
    r = df.loc[tid]
    print("="*120)
    print("task_id:", tid)
    print("domain:", r.get("off_domain"), "Collection:", r.get("off_Collection"))
    print("\nSCORES:")
    print(f"  final off={r['off_final_hmean']:.6f}  evi={r['evi_final_hmean'] if pd.notna(r['evi_final_hmean']) else None}  delta={r['delta_final']}")
    print(f"  RL_F   off={r['off_RL_F']}  evi={r['evi_RL_F']}  delta={r['delta_RL_F']}")
    print(f"  RB_llm off={r['off_RB_llm']}  evi={r['evi_RB_llm']}  delta={r['delta_RB_llm']}")
    print(f"  RB_alg off={r['off_RB_alg']}  evi={r['evi_RB_alg']}  delta={r['delta_RB_alg']}")

    inp = r.get("off_input")
    if inp is not None:
        print("\nINPUT (truncated):")
        print(str(inp)[:2000])

    print("\nOFFICIAL ANSWER:\n", (r.get("off_pred") or "")[:ans_chars])
    print("\nEVIDENCE ANSWER:\n", (r.get("evi_pred") or "")[:ans_chars])

    oc = off_claims.get(tid, {}).get("RL_F_claims")
    ec = evi_claims.get(tid, {}).get("RL_F_claims")
    o_flat = flatten_claims(oc)
    e_flat = flatten_claims(ec)

    print("\nRL_F CLAIMS summary:")
    print("  off:", summarize_labels(o_flat), " n=", len(o_flat))
    print("  evi:", summarize_labels(e_flat), " n=", len(e_flat))

    # 打印 evidence 里最可疑的（unsupported/unverifiable）claims
    bad = [x for x in e_flat if x.get("label") in ("unsupported","unverifiable")]
    print("\nEVIDENCE bad claims (top):")
    for i, c in enumerate(bad[:max_claims], 1):
        print(f"{i:02d}. [{c.get('label')}] {c.get('claim')}")
        ev = c.get("evidence","")
        if ev:
            print("    ev:", str(ev)[:200])

# 打印最差3个
for tid in worst3.index:
    print_case(tid)


task_id: be461bfeda2d4826cdb663dcaa7d1ced<::>4
domain: None Collection: mt-rag-ibmcloud-elser-512-100-20240502

SCORES:
  final off=nan  evi=None  delta=nan
  RL_F   off=0.0  evi=0.0  delta=0.0
  RB_llm off=0.2  evi=0.2  delta=0.0
  RB_alg off=0.2343714028  evi=0.2767873447  delta=0.042415941900000004

INPUT (truncated):
[{'speaker': 'user', 'text': 'What are the steps to be taken to gather the relevant worker node data?', 'metadata': {'author_type': 'human', 'author_id': 'fd309ff8-8b41-49a0-b483-2fee97472e71', 'created_at': 1724165026}}, {'speaker': 'agent', 'text': "   There are general steps to gather the relevant worker node data: First,  Check the conditions of your worker nodes and cluster before you gather data. This includes checking the CPU and memory level of your nodes. If any node is over 80% in either CPU or memory usage, consider provisioning more nodes or reducing your workload.Secondly if all worker nodes in a cluster, or in a single zone, subnet, or VLAN are affected, 

In [8]:
import json

tids = list(worst3.index)
subset = []

# 从 eval_input_evidence.jsonl 把对应样本抽出来
inp_path = "eval_input_evidence.jsonl"
with open(inp_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): 
            continue
        r=json.loads(line)
        if r.get("task_id") in tids:
            subset.append(r)

with open("worst3_eval_input.jsonl","w",encoding="utf-8") as w:
    for r in subset:
        w.write(json.dumps(r, ensure_ascii=False) + "\n")

len(subset)


3

In [ ]:
python3 build_prompts_train.py \
  --taska_file worst3_eval_input.jsonl \
  --cleaned_root cleaned_dataset \
  --task_name concat_lastturn_rewrite_gpt \
  --out_csv  outputs/taskc/prompts_worst3_v4.csv \
  --prompt_style v4 \
  --topk 5 \
  --max_doc_chars 1200 \
  --domains "clapnq,cloud,fiqa,govt" \
  --add_ctx_count \
  --add_question_source

usage: ipykernel_launcher.py [-h] -i INPUT -o OUTPUT [--rb_alg_key RB_ALG_KEY]
                             [--out_key OUT_KEY]
ipykernel_launcher.py: error: the following arguments are required: -i/--input, -o/--output


SystemExit: 2

/home/smen/.conda/envs/mtrag310/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [17]:
import pandas as pd
import numpy as np

df = pd.read_json(
    "outputs/taskc/gen_eval_official.jsonl",
    lines=True
)

def get_metric(row, name):
    try:
        return row["metrics"][name][0]
    except:
        return np.nan

df["HM"] = df.apply(lambda r: get_metric(r, "HM_RL_RBllm_RBalg"), axis=1)

print("Mean HM:", df["HM"].mean())
print("Median HM:", df["HM"].median())


Mean HM: nan
Median HM: nan


In [9]:
import json

# path = "outputs/taskc/gen_eval_v4_rewrite_gpt.jsonl"   # 你改成你的文件
path = "outputs/taskc/gen_eval_v4.jsonl"
out_key = "HM_RLFRBagg"               # 你想叫啥都行

def get1(x):
    # metrics 里通常是 [val]
    if x is None:
        return None
    if isinstance(x, list):
        return float(x[0]) if x else None
    return float(x)

def harmonic_mean(a, b, c):
    # 任意一个缺失或<=0 -> 给 0（你也可以改成 None）
    vals = [a, b, c]
    if any(v is None or v <= 0 for v in vals):
        return 0.0
    return 3.0 / (1.0/a + 1.0/b + 1.0/c)

hms = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        j = json.loads(line)
        m = j.get("metrics", {}) or {}

        rl_f   = get1(m.get("RL_F"))
        rb_llm = get1(m.get("RB_llm"))
        rb_agg = get1(m.get("RB_agg"))   # 你说的 RB_alg 就用这个

        hm = harmonic_mean(rl_f, rb_llm, rb_agg)
        # print(hm)
        hms.append(hm)

print("N =", len(hms))
print("mean HM =", sum(hms)/len(hms))
print("first 5 =", hms[:5])


N = 100
mean HM = 0.5590977115279624
first 5 = [0.7393996244356111, 0.9369150579164136, 0.620036552700709, 0.759180111734804, 0.7069295195247832]


In [20]:
import json

path = "outputs/taskc/gen_eval_v4.jsonl"
n = 0
ok_rl = ok_rbllm = ok_rbag = ok_all = 0

def ok(x):
    if x is None: return False
    if isinstance(x, list): return len(x)>0 and x[0] is not None
    return True

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        n += 1
        m = json.loads(line).get("metrics", {}) or {}
        a = m.get("RL_F"); b = m.get("RB_llm"); c = m.get("RB_agg")
        ok_rl += ok(a); ok_rbllm += ok(b); ok_rbag += ok(c)
        ok_all += (ok(a) and ok(b) and ok(c))

n, ok_rl, ok_rbllm, ok_rbag, ok_all


(100, 100, 100, 100, 100)

In [ ]:
import pandas as pd

df = pd.read_csv('outputs/taskc/prompt_official_rewrite_gpt_ans.csv')
df.isna().sum()

task_id                0
conversation_id        0
Collection             0
domain                 0
prompt                 0
n_contexts             0
question_source        0
raw_answer             0
raw_response_json      0
model                  0
temperature            0
error                507
dtype: int64

In [6]:
df['raw_answer'].sample(10).values

array(['Emergency kit recommendations for ensuring readiness include having supplies to last at least three days. A basic kit should contain one gallon of water per person per day, a three-day supply of non-perishable food, a battery-powered or hand-crank radio (like a NOAA Weather Radio), a flashlight with extra batteries, a list of medications, and a dust mask. It is also recommended to consider the specific needs of your household, including pets, dietary requirements, medical equipment, and prescriptions. For your vehicle, ensure you have a properly inflated spare tire and the tools to change it.',
       'Based on the documents, a Discover card can be a good option in specific situations, but it has limitations. It is beneficial for balance transfers with introductory 0% APR offers and offers a unique feature for getting cash over at checkout without a fee. Historically, it had lower merchant fees. However, for worldwide travel, Discover is less appealing than Visa or Mastercard d